# Crypto trend filter (BTC, ETH)

Single-asset trend following on the two largest crypto majors: hold the coin while its price is above a long-window moving average, sit in cash otherwise. The point is to harvest crypto's upside while sidestepping its multi-year drawdowns.

**Retail fit.** BTC-USD / ETH-USD are spot tickers on yfinance; the only execution decision is on/off, no shorting, no leverage. Crypto trades 24/7, so we use a calendar-day index from yfinance and rebalance weekly to keep turnover sensible.

**Diagnostic / tradable.** The trend signal is lagged inside `run_backtest`, so today's signal earns tomorrow's return. Buy-and-hold benchmarks are computed from the same price stream.

In [ ]:
import numpy as np
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

UNIVERSE = ["BTC-USD", "ETH-USD"]
START = "2017-01-01"
END = None
SMA_WINDOW = 200       # calendar-day SMA on the close
REBAL = "W-FRI"        # weekly rebalance limits whipsaw cost
COSTS_BPS = 5.0        # retail crypto exchange spread/commission, one-way
SLIPPAGE_BPS = 10.0    # generous for retail liquidity

In [ ]:
prices = load_prices(UNIVERSE, start=START, end=END).dropna(how="all").ffill()
prices.tail()

In [ ]:
# Trend signal: 1 when price > SMA(200), else 0. Per-asset, equal-weight when on.
sma = prices.rolling(SMA_WINDOW, min_periods=SMA_WINDOW).mean()
signal = (prices > sma).astype(float)

# Equal-weight across whichever coins are on; missing prices -> 0.
active = signal.where(prices.notna(), 0.0)
weights = active.div(len(UNIVERSE))   # max gross 100% / N
weights.tail()

In [ ]:
res = run_backtest(weights, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)

# Benchmarks: buy-and-hold each coin, and a 50/50 buy-and-hold blend.
rets = prices.pct_change().fillna(0.0)
bh_blend = fixed_weight_returns(rets, {t: 1.0 / len(UNIVERSE) for t in UNIVERSE})

streams = pd.DataFrame({
    "trend (BTC+ETH/2)": res.returns,
    "BH blend (BTC+ETH/2)": bh_blend,
    "BH BTC": rets["BTC-USD"],
    "BH ETH": rets["ETH-USD"],
}).dropna(how="all")
stats = pd.DataFrame({name: pd.Series(summary(s)) for name, s in streams.items()})
stats

In [ ]:
# % of days the trend filter was 'on' for each coin -- a sanity check on signal coverage.
active.mean().rename("pct_days_long")

In [ ]:
# Round-trips per year and average cost drag give a feel for how 'busy' the strategy is.
annual_turnover = res.turnover.resample("YE").sum()
annual_costs = res.costs.resample("YE").sum()
pd.DataFrame({"turnover_per_year": annual_turnover, "cost_drag_per_year": annual_costs})

In [ ]:
equity_curve(streams)

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)

## Read

- Goal: capture crypto's positive trends and stay out during bear quarters (2018, 2022).
- Expect lower CAGR than buy-and-hold but materially better drawdown / Calmar.
- Whipsaws are the cost — see `pct_days_long` and `turnover_per_year`. Bumping SMA to 250 or rebalancing only monthly will cut activity further at the price of slower re-entries.
- All-in implementation: one Coinbase / Kraken account, weekly on Friday close, two buy/sell decisions.